# Week 1 — Problem Definition

This notebook does two things:

1. States, in plain language, **what problem this project solves** and why it is
   a supervised, multiclass classification problem.
2. Takes the **first look at the raw data** — shape, columns, dtypes, and a few
   rows — using the same loader that the test suite exercises.

Nothing is modelled, plotted or preprocessed here. Exploration is Week 2,
preparation is Week 3, models start in Week 4.

Reference reading: [`docs/curriculum/week01/learning_notes.md`](../docs/curriculum/week01/learning_notes.md).

## 1. The problem

> Given the measurable growing conditions of a plot of land — soil nutrients,
> weather and rainfall — recommend the crop most suited to it.

```
Input:  N=90, P=42, K=43, temperature=25, humidity=80, ph=6.5, rainfall=200
Output: rice
```

**Why supervised?** Every historical record in the dataset already carries the
correct answer in its `label` column. We learn the mapping from conditions to
crop from those answered examples.

**Why classification, not regression?** The thing predicted is a *category*
(one of 22 crop names), not a quantity on a continuous scale. There is no
meaningful sense in which a prediction can be "slightly" wrong: you either
named the right crop or you did not. And because there are more than two
categories, it is **multiclass** classification specifically.

**Why not hand-written rules?** Seven variables that interact (rice tolerates
less rainfall when humidity is high; potassium needs shift with pH), thresholds
that no agronomist can state to one decimal place, and 22 crops to cover. The
numbers exist in the data, not in anyone's head.

## 2. Loading the data

We import `load_data` from `src/` rather than calling `pd.read_csv` here. Two
reasons:

* the loader resolves an **absolute** path from its own file location, so it
  works from `notebooks/` just as it does from the repository root;
* it runs the **dataset contract** (`validate_dataset`) immediately after
  reading, so anything below this cell is guaranteed to be operating on data of
  the expected shape, columns, dtypes and label set.

The `sys.path` line exists because this notebook lives one directory below the
repository root, and `src` is imported as a top-level package.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.data import EXPECTED_LABELS, FEATURE_COLUMNS, TARGET_COLUMN, load_data

print("pandas version:", pd.__version__)

pandas version: 2.2.3


In [2]:
crops = load_data()   # reads the CSV and validates it in one step
type(crops)

pandas.core.frame.DataFrame

### Why a pandas `DataFrame` and not lists or dicts?

Plain Python could hold this file as a list of dictionaries. The CSV is only
2,200 rows, so it would even fit comfortably in memory. The problem is
everything that comes *after* loading:

| Question | With a `DataFrame` | With lists/dicts |
| --- | --- | --- |
| How many rows and columns? | `df.shape` | `len(rows)`, plus a loop over keys |
| Are any values missing? | `df.isna().sum()` | nested loop, per column |
| How many rows per crop? | `df["label"].value_counts()` | manual `Counter` |
| Mean rainfall per crop? | `df.groupby("label")["rainfall"].mean()` | dict of lists, then a loop |

A `DataFrame` is a table of rows and **named, typed columns**. The types matter:
pandas knows `rainfall` is a float and `label` is text, so it can reject
nonsense arithmetic and store the numbers compactly. It is also the format
every library we use later — matplotlib, seaborn, scikit-learn — expects.

## 3. Shape: how much data is there?

In [3]:
crops.shape

(2200, 8)

2,200 rows and 8 columns: seven features plus one label. Small by machine
learning standards — small enough that everything in this course runs in
seconds on a laptop, and small enough that the file is committed to git so
everyone trains on byte-identical data.

## 4. First rows

In [4]:
crops.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


## 5. Structure and dtypes

In [5]:
crops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB


Three things to read off that output:

* **2,200 non-null in every column** — no missing values anywhere. Real
  agricultural measurements are rarely this clean, which is itself worth being
  suspicious about (see exercise C4).
* **dtypes** — `N`, `P`, `K` are integers; `temperature`, `humidity`, `ph`,
  `rainfall` are floats; `label` is `object`, which is how pandas stores text.
* **memory usage** — a few hundred kilobytes. Nothing here needs special
  handling for size.

## 6. The seven features and the label

Each row is one **instance**: a set of measured conditions plus the crop that
suits them.

| Column | What it physically measures | Unit | Plausible range |
| --- | --- | --- | --- |
| `N` | Nitrogen content of the soil — drives leaf and stem growth | ratio index | ~0–140 |
| `P` | Phosphorus content — root development and flowering | ratio index | ~5–145 |
| `K` | Potassium content — water regulation and disease resistance | ratio index | ~5–205 |
| `temperature` | Average air temperature during the growing period | °C | ~8–44 |
| `humidity` | Average relative humidity | % (0–100) | ~14–100 |
| `ph` | Soil acidity/alkalinity | pH scale (0–14, 7 neutral) | ~3.5–10 |
| `rainfall` | Total rainfall over the growing period | mm | ~20–300 |
| `label` | **Target.** The crop suited to those conditions | crop name | 22 values |

`N`, `P` and `K` are reported by the source dataset as unitless soil-test index
values, not as kilograms per hectare, so they are comparable to one another
within this dataset but not directly to a lab report. Ranges above are the
values the dataset happens to contain; Week 2 confirms them statistically
rather than by assertion.

In [6]:
print("Features (7):", list(FEATURE_COLUMNS))
print("Target       :", TARGET_COLUMN)

Features (7): ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
Target       : label


## 7. The label set

Week 1 records the exact set of 22 crop names as part of the dataset contract.
Every later week that encodes, filters or predicts labels must match this set —
a new or misspelled crop should fail loudly, not quietly change what the model
is trained to do.

In [7]:
observed = sorted(crops[TARGET_COLUMN].unique())
print(len(observed), "crops")
print(observed)
print("matches the recorded contract:", set(observed) == set(EXPECTED_LABELS))

22 crops
['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']
matches the recorded contract: True


In [8]:
crops[TARGET_COLUMN].value_counts()

label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64

Exactly 100 rows per crop — a perfectly balanced dataset. That is convenient
(no class imbalance to handle) and slightly suspicious (real data is never this
tidy). Week 2 investigates.

## 8. What the validation buys us

`load_data()` called `validate_dataset()` before returning. Had the CSV been
swapped for a file with a renamed column, a stray space in a header, a text
value in a numeric column, a truncated row count or an unfamiliar crop name,
this notebook would have stopped at the load cell with a message naming the
exact problem — instead of failing three weeks later as a `KeyError` inside
preprocessing or a silent label mismatch during training.

In [9]:
from src.data import DatasetValidationError, validate_dataset

broken = crops.rename(columns={"ph": " ph"})   # one stray space in a header

try:
    validate_dataset(broken)
except DatasetValidationError as error:
    print(type(error).__name__)
    print(error)

DatasetValidationError
Unexpected columns. Expected ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label'], got ['N', 'P', 'K', 'temperature', 'humidity', ' ph', 'rainfall', 'label']. Stray whitespace in a header (' ph') counts as a different name.


## 9. Where this leaves us

We can state the problem precisely and load the data with confidence that it is
the data we think it is.

We still know nothing about what the numbers *look like*: how features are
distributed, whether crops overlap, whether anything is anomalous. That is
**Week 2 — exploratory data analysis**.